In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
df = pd.read_csv(os.path.join(path, "Q1_data.csv"))

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
plt.hist(df["Delivery_Time"].dropna(), bins=30, edgecolor='black', color='blue')
plt.title(f"Target Distribution")
plt.xlabel("Delivery Time")
plt.ylabel("Frequency")

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])
df.head()

In [ ]:
# Task 2: Write your code here:

# First let's inspect columns with missing values

missing_values = df.isnull().sum()
missing_values = missing_values[missing_values>0]
print(missing_values)
#Task Continued Below

In [ ]:
# Let's look at the data types of these columns
for col in missing_values.index:
  print(f"{col}'s Data type : {df[col].dtype}")

  # Task continued below

In [ ]:
# First rows with key features (and target) missing
df.dropna(subset=['Traffic_Level', 'Time_of_Day', 'Delivery_Time'], inplace=True)
df.isnull().sum()
# Task continued below

In [ ]:
# Now filling the weather with the mode and the experience years with the mean

df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())
print(f"{df.isnull().sum().sum()} Missing Values Remaining")
df.head()

In [ ]:
# Task 3: Write your code here:
duplicate_count = df.duplicated().sum()
if duplicate_count>0:
  df.drop_duplicates(inplace=True)
  print("Duplicates Removed")
else:
  print("No duplicates found")

In [ ]:
# Task 4: Write your code here:

# Let's first check the categoricla columns in the dataset
categorical_cols = df.select_dtypes(include=['object']).columns
print(categorical_cols)

# Task continued below

In [ ]:
# Now let's choose the encoders based on them
# Traffic Level is ordinal -> use label encoder
# Vehicle type, Weather, Time of day are nominal -> use one hot encoder

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

ordinal = ['Traffic_Level']
nominal = ['Vehicle_Type', 'Weather', 'Time_of_Day']

for ord in ordinal:
  le = LabelEncoder()
  df[ord] = le.fit_transform(df[ord])

for nom in nominal:
  onehot_encoder = OneHotEncoder(sparse_output=False)
  categories = pd.DataFrame(df[nom])
  df[nom] = onehot_encoder.fit_transform(categories)

df[categorical_cols].head()

In [ ]:
df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
features = df.drop(columns='Delivery_Time').columns
targets = df['Delivery_Time']

df[features] = scaler.fit_transform(df[features])

df.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:

X = df.drop(columns=["Delivery_Time"])
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:

# Using KFold since we are dealing with a regressin task
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestRegressor(n_estimators=200)

train_losses = []

for fold, (train_idx, test_idx) in enumerate(kfold.split(X), start=1):
  print(f"\nTraining Fold {fold}/5..]\n")

  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

  rf.fit(X_train, y_train)

  y_pred = rf.predict(X_test)

  mse = mean_squared_error(y_test, y_pred)

  train_losses.append(mse)

avg_loss = np.mean(np.array(train_losses))
#Task continued below

In [ ]:
print(f"Average MSE Across All Folds: {avg_loss}")

In [ ]:
# Task 1: Write your code here:

importances = rf.feature_importances_

features = X.columns

sorted_idx = np.argsort(importances)

plt.barh(features[sorted_idx], importances[sorted_idx])
plt.xlabel("Features")
plt.ylabel("Importance")
plt.title("Feature Importances")

In [ ]:
# Task 2: Write your code here:

predicted_time = rf.predict(X)
plt.hist(predicted_time, bins=30, edgecolor='black')
plt.xlabel("Predicted Delivery Time")
plt.ylabel("Frequency")
plt.title("Predicted Delivery Time Distribution")

In [ ]:
# Task Bonus: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

model1 = RandomForestRegressor(n_estimators=200)
model2 = CatBoostRegressor(verbose=0)



In [ ]:

# Using KFold since we are dealing with a regressin task
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

train_losses = []

for fold, (train_idx, test_idx) in enumerate(kfold.split(X), start=1):
  print(f"\nTraining Fold {fold}/5..]\n")

  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

  model1.fit(X_train, y_train)
  model2.fit(X_train, y_train)

  y_pred1 = model1.predict(X_test)
  y_pred2 = model2.predict(X_test)

  avg_pred = (y_pred1+y_pred2)/2
  mse = mean_squared_error(y_test, avg_pred)

  train_losses.append(mse)

avg_loss = np.mean(np.array(train_losses))
#Task continued below

In [ ]:
print(f"MSE of Average Predictions: {avg_loss}")

In [ ]:
# Better than Random Forest alone! :)